In [19]:
import Pkg; 

if split(pwd(),"/")[end] == "config_generators"
    cd(joinpath(@__DIR__, "../../"))
    Pkg.activate("Project.toml")
end

using MorphoMol

In [20]:
mol_type = "small_icosahedron"

template_centers = MorphoMol.TEMPLATES[mol_type]["template_centers"]
template_radii = MorphoMol.TEMPLATES[mol_type]["template_radii"]

x_init = "Vector{Float64}([])"
bnds = 200.0
delaunay_eps = 100.0
overlap_jump = 0.0
overlap_slope = 1.1
rs = 1.4
η = 0.3665
T = 0.0

prefactors = MorphoMol.Energies.get_prefactors(rs, η)
comment = "scan"
comment = replace(comment, " " => "_")
simulation_time_minutes = 24.0 * 60.0
energies = [:tasp, :dbbasp]
perturbation = :single_random

n_mols = [12]
σ_rs = [0.25]
σ_ts = [1.25]
persistence_weights = [[1.0, p1, p2] for p1 in -2.0:0.5:2.0 for p2 in -2.0:0.5:2.0]

inputs = []

for energy in energies
    for n_mol in n_mols
        for σ_r in σ_rs
            for σ_t in σ_ts
                for pws in persistence_weights
                    input = Dict(
                        "energy" => energy,
                        "perturbation" => perturbation,
                        "mol_type" => mol_type,
                        "template_centers" => template_centers,
                        "template_radii" => template_radii,
                        "n_mol" => n_mol,
                        "x_init" => x_init,
                        "comment" => comment,
                        "bnds" => bnds,
                        "rs" => rs,
                        "η" => η,
                        "σ_r" => σ_r,
                        "σ_t" => σ_t,
                        "T" => T,
                        "persistence_weights" => pws,
                        "overlap_jump" => overlap_jump,
                        "overlap_slope" => overlap_slope,
                        "delaunay_eps" => delaunay_eps,
                        "simulation_time_minutes" => simulation_time_minutes
                    )
                    push!(inputs, input)
                end
            end
        end
    end
end

In [21]:
open("julia_scripts/input_data/inputs.jl", "w") do file
    write(file, "input_templates = $(inputs)")
end

116643

In [22]:
input_specifier = "generic_$(mol_type)_$(comment)"

"generic_small_icosahedron_scan"

In [23]:
total_simulations = length(inputs)

162

In [26]:
hours = Int(round(simulation_time_minutes / 60.0))
days = hours ÷ 24
remaining_hours = hours % 24
remaining_hours_string = remaining_hours < 10 ? "0$(remaining_hours)" : string(remaining_hours)
buffer_time_string = simulation_time_minutes < 5 ? "0$(Int(simulation_time_minutes)+2)" : "30"

open("hpc_scripts/$(input_specifier)_script.job", "w") do io
    println(io, "#!/bin/bash")
    println(io, "#SBATCH --job-name=SolvationSimulations")
    println(io, "#SBATCH --time=0$(days)-$(remaining_hours_string):$(buffer_time_string)")
    println(io, "#SBATCH --ntasks=1")
    println(io, "#SBATCH --cpus-per-task=1")
    println(io, "#SBATCH --array=1-$(total_simulations)")
    println(io, "#SBATCH --chdir=/work/spirandelli/MorphoMolHPC/")
    println(io, "#SBATCH -o ./job_log/$(input_specifier)/%a.out # STDOUT")
    println(io, "")
    println(io, "export http_proxy=http://proxy2.uni-potsdam.de:3128 #Setting proxy, due to lack of Internet on compute nodes.")
    println(io, "export https_proxy=http://proxy2.uni-potsdam.de:3128")
    println(io, "")
    println(io, "module purge")
    println(io, "module load lang/Julia/1.7.3-linux-x86_64")
    println(io, "")
    println(io, "julia -e \"$(escape_string("include(\"julia_scripts/generic_call.jl\"); generic_call(\$SLURM_ARRAY_TASK_ID)"))\"")
    println("Done writing job script.")
end

Done writing job script.
